# Miniproyecto 1 — De la bolsa de palabras a los transformers

## Clasificación de polaridad en reseñas turísticas en español

**Maestría · Universidad Icesi · Procesamiento de Lenguaje Natural**

Autores: Aguado · Cruz · Ramírez

---

> **Cómo leer este notebook.** Cada sección de modelado va precedida de la pregunta que
> intenta responder y cerrada con lo que efectivamente encontramos. Las gráficas no están
> de adorno: cada una induce una decisión posterior, y esa cadena se hace explícita en la
> síntesis del análisis exploratorio (§3.8).

## 0. El problema

<!-- REDACTAR (Fase 1):
     - Enunciado en un párrafo: qué se predice y a partir de qué.
     - Por qué NO es un problema resuelto de análisis de sentimiento. Los cuatro
       argumentos están en docs/SPEC.md §1.2: ordinalidad, desbalance, sesgo geográfico
       y vocabulario de dominio. Escribirlos con voz propia, no copiarlos.
     - Por qué la escalera de cuatro representaciones es la propuesta adecuada (SPEC §1.3).
     - Qué se hereda de los notebooks guía 3 y 4, y qué es aporte propio. Ser explícito:
       la consigna valora la originalidad y la rúbrica la puntúa.
-->

### Mapa del notebook

| Sección | Contenido |
|---|---|
| 1 | Entorno, reproducibilidad y configuración adaptativa |
| 2 | Carga del corpus |
| 3 | Análisis exploratorio — y las decisiones que induce |
| 4 | Preprocesamiento, tokenización y protocolo experimental |
| 5–8 | Los cuatro modelos: TF-IDF, LSTM, BiLSTM+spaCy, BETO |
| 9–12 | Extensiones propias: ordinalidad, geografía, atención, embeddings |
| 13 | Tarea de control: predecir el tipo de establecimiento |
| 14 | Comparación global y análisis de errores |
| 15 | Conclusiones y limitaciones |

---

## 1. Entorno y reproducibilidad

Antes de cualquier análisis fijamos las condiciones que hacen que este notebook produzca los
mismos números en cada ejecución, y que corra igual en Colab con GPU, en Colab sin GPU y en
una máquina local.

<!-- REDACTAR: por qué la reproducibilidad no es un trámite sino parte del resultado:
     sin semilla fija, comparar cuatro modelos no significa nada. -->

In [ ]:
import sys, os, warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ejecutando en Colab: {IN_COLAB}')
print(f'Python: {sys.version.split()[0]}')

Instalamos únicamente lo que falte. En Colab casi todo viene preinstalado; el modelo de
vectores de spaCy en español es la descarga pesada (~570 MB) y solo se necesita a partir de
la sección 7.

In [ ]:
# Instalación de dependencias (una sola celda para todo el notebook).

# TODO (Fase 1 — pip install condicional + spacy download es_core_news_lg)

### Semilla global

Fijamos `SEED = 42` en todas las fuentes de aleatoriedad del pipeline: submuestreo,
particiones, inicialización de pesos y orden de los lotes.

In [ ]:
SEED = 42


def fijar_semilla(seed: int = SEED):
    \"\"\"Fija la semilla en todas las librerías que introducen aleatoriedad.\"\"\"
    ...

# TODO (Fase 1 — random, numpy, torch, torch.cuda; devolver nada, imprimir confirmación)

### Configuración adaptativa

El notebook debe correr en cualquier entorno, no fallar en el que no tenga GPU. Esta celda
detecta el dispositivo y ajusta tamaños de lote, número de épocas y estrategia para el
transformer. La configuración activa se imprime para que quede registrada junto a los
resultados.

In [ ]:
# Detección de dispositivo y tabla de configuración (CFG).
# Debe imprimir: dispositivo, nombre de GPU si existe, y la configuración resultante.

# TODO (Fase 1 — CFG con batch_size, epochs, subset_size, estrategia BETO)

---

## 2. El corpus

Trabajamos con [`vg055/Rest-Mex2025`](https://huggingface.co/datasets/vg055/Rest-Mex2025):
208,051 reseñas turísticas en español sobre destinos de México, del shared task
**Rest-Mex 2025** de IberLEF. Licencia CC-BY-4.0.

<!-- REDACTAR: por qué este corpus y no otro (docs/DECISIONS.md §D-001), y qué lo hace
     apropiado para las técnicas que vamos a comparar. -->

In [ ]:
# Carga desde HuggingFace y conversión a pandas.

# TODO (Fase 1 — load_dataset + to_pandas + shape/dtypes/head)

Antes de cualquier estadística, conviene ver el material crudo. Estas tres reseñas —una
negativa, una intermedia y una positiva— dan la medida del registro, la extensión y el tipo
de lenguaje al que nos enfrentamos.

In [ ]:
# Mostrar 3 reseñas completas de polaridad 1, 3 y 5.

# TODO (Fase 1)

<!-- REDACTAR: qué se observa en los ejemplos. Registro coloquial, mezcla de temas dentro
     de una misma reseña, anglicismos, uso de mayúsculas y signos. Anticipar aquí que la
     reseña de 3 estrellas probablemente contenga elogios y quejas mezclados: eso explicará
     buena parte de los errores de la sección 14. -->

---

## 3. Análisis exploratorio

Esta sección no es un requisito administrativo: de aquí salen cuatro decisiones concretas
—la métrica principal, la longitud máxima de secuencia, el diseño de la partición
geográfica y el tratamiento de la ordinalidad— y todas quedan justificadas con datos en
§3.8.

El análisis se hace sobre **el corpus completo** (208k). Solo el entrenamiento usará una
submuestra.

### 3.1 Calidad de los datos

In [ ]:
# Nulos, duplicados exactos, reseñas vacías o degeneradas, rangos de valores.

# TODO (Fase 2)

<!-- LEER: qué problemas hay, cuáles importan, qué se decide hacer con ellos. -->

### 3.2 Distribución de la polaridad

El primer hallazgo, y el que reorienta todo el trabajo.

In [ ]:
# Barras con conteo y porcentaje por estrella.

# TODO (Fase 2)

In [ ]:
# Cálculo EXPLÍCITO del baseline de clase mayoritaria: accuracy y macro-F1.

# TODO (Fase 2)

<!-- LEER: el desbalance y su consecuencia. Un modelo que responda siempre "5" alcanza
     ~65.6% de accuracy sin leer nada. Nombrar aquí la decisión que esto fuerza: macro-F1
     como métrica principal, y accuracy solo reportado junto a este baseline. -->

### 3.3 Tipo de establecimiento, región y pueblo

In [ ]:
# Distribuciones de Type, Region y Town.

# TODO (Fase 2)

<!-- LEER: el contraste entre un `Type` balanceado y una polaridad que no lo está —de ahí
     sale la tarea de control de la §13— y la concentración geográfica en Quintana Roo,
     que motiva la partición por región de la §10. -->

### 3.4 Longitud de las reseñas → de dónde sale `MAX_LEN`

Los notebooks guía fijan la longitud de secuencia en 256 o 512 tokens sin justificarlo.
Aquí la derivamos de la distribución real: `MAX_LEN` será el percentil 95 en tokens, y
reportaremos qué fracción de reseñas queda truncada.

In [ ]:
# Longitudes en caracteres y en tokens: histograma, percentiles, y fijación de MAX_LEN.

# TODO (Fase 2 — MAX_LEN = P95 redondeado; imprimir tasa de truncamiento)

<!-- LEER: la asimetría de la distribución y por qué el P95 es el compromiso correcto. -->

### 3.5 Léxico distintivo por polaridad

Las frecuencias crudas están dominadas por palabras funcionales y no distinguen nada. Usamos
**log-odds ratio con prior de Dirichlet**, que mide qué tan sobrerrepresentado está un
término en una clase respecto al resto del corpus y es robusto ante términos poco frecuentes.

In [ ]:
# Log-odds ratio con prior informativo; términos más distintivos de 1★ y de 5★.

# TODO (Fase 2)

<!-- LEER: qué palabras marcan cada extremo. Guardar esta lista: en la §11 comprobaremos si
     el mecanismo de atención llega por su cuenta a los mismos términos. -->

### 3.6 Interacciones

In [ ]:
# Polaridad × tipo (heatmap), polaridad × región, longitud × polaridad (boxplot).

# TODO (Fase 2)

<!-- LEER: ¿son los hoteles más criticados que las atracciones? ¿escriben más largo quienes
     se quejan? Si la longitud correlaciona con la polaridad, es una señal que el modelo
     puede explotar y que conviene tener presente al interpretar resultados. -->

### 3.7 Cobertura del vocabulario frente a los embeddings preentrenados

El modelo 3 inicializará sus embeddings con vectores de spaCy entrenados sobre español
general. Antes de hacerlo, medimos cuánto de nuestro vocabulario turístico y regional está
efectivamente cubierto.

In [ ]:
# % de tipos y de tokens del corpus con vector en es_core_news_lg; OOV más frecuentes.

# TODO (Fase 2)

<!-- LEER: la tasa de cobertura y los OOV de dominio. Si la cobertura es baja, es un
     resultado por derecho propio y anticipa el desempeño de la §7. -->

### 3.8 Síntesis: del hallazgo a la decisión

<!-- REDACTAR — la celda más importante del EDA. Una lista numerada, cada línea con la
     forma «hallazgo → decisión → dónde se aplica»:

     1. Desbalance 65/22/7/3/3  →  macro-F1 como métrica principal  →  §4, todas las tablas
     2. Distribución de longitudes  →  MAX_LEN = P95  →  §4
     3. Concentración en Quintana Roo  →  partición geográfica  →  §10
     4. Objetivo ordinal  →  MAE y QWK + formulaciones ordinales  →  §9
     5. Cobertura de embeddings  →  expectativa sobre el modelo 3  →  §7
     6. `Type` balanceado  →  tarea de control  →  §13
-->

---

## 4. Preprocesamiento y protocolo experimental

Para que la comparación entre cuatro modelos signifique algo, todos deben ver exactamente
los mismos datos, la misma partición y las mismas métricas. Esta sección fija ese contrato.

### 4.1 Tokenización

Partimos del tokenizador del notebook guía y corregimos dos cosas: construimos el vocabulario
**por frecuencia** (el notebook 4 lo hace por orden de aparición, ver `docs/DECISIONS.md`
§D-002) y verificamos que la normalización preserve tildes y ñ.

In [ ]:
# Tokenizador + construcción de vocabulario con most_common.

# TODO (Fase 3)

In [ ]:
# Verificación ida y vuelta sobre una reseña real; tasa de [UNK] y de truncamiento.

# TODO (Fase 3)

<!-- LEER: tamaño del vocabulario, tasa de UNK, qué se pierde al truncar. -->

### 4.2 Submuestra y particiones

Entrenamos sobre 40,000 reseñas estratificadas por polaridad. **No balanceamos las clases**:
el desbalance es una propiedad del dominio que queremos estudiar, no un defecto que ocultar.

In [ ]:
# Submuestra estratificada (40k, SEED) + verificación de que se preservan las proporciones.

# TODO (Fase 3)

In [ ]:
# Split A: aleatorio estratificado 80/10/10.

# TODO (Fase 3)

In [ ]:
# Split B: geográfico, regiones held-out. Documentar qué regiones y por qué.

# TODO (Fase 3 — registrar la elección en docs/DECISIONS.md)

### 4.3 Métricas y baselines

Una única función de evaluación que devuelve las seis métricas, usada sin excepción por
todos los modelos del notebook. Es lo que hace que la tabla final sea comparable.

In [ ]:
def evaluar(y_true, y_pred, nombre=''):
    \"\"\"macro-F1, accuracy, MAE, QWK, F1 por clase.\"\"\"
    ...

# TODO (Fase 3)

In [ ]:
# Baselines: clase mayoritaria y azar estratificado.

# TODO (Fase 3)

<!-- LEER: el piso que todo modelo debe superar para ser considerado útil. -->

---

## 5. Modelo 1 — TF-IDF + Regresión Logística

Empezamos por el baseline clásico, y no por trámite: en clasificación de sentimiento
con textos de longitud media, TF-IDF con n-gramas es un rival duro. Si las redes neuronales
no lo superan, es que no están aportando nada sobre esta tarea, y eso también sería un
hallazgo.

Usamos n-gramas de 1 a 2 y `class_weight='balanced'` para que la regresión no colapse sobre
la clase mayoritaria.

In [ ]:
# Vectorización TF-IDF + entrenamiento de la regresión logística.

# TODO (Fase 4)

In [ ]:
# Evaluación con evaluar() + coeficientes más pesados por clase.

# TODO (Fase 4)

<!-- LEER: cómo se compara con los baselines, y qué términos pesan más por clase.
     Contrastar con el léxico distintivo del EDA §3.5. -->

---

## 6. Modelo 2 — LSTM entrenada desde cero

Réplica de la arquitectura del notebook 3 del curso —`Embedding → LSTM → Linear`—
pero sobre nuestros datos. Es el punto de comparación directo con el material de clase: los
embeddings se aprenden desde cero, sin conocimiento lingüístico previo, solo a partir de las
40,000 reseñas de entrenamiento.

La pregunta es si añadir orden secuencial y representaciones densas compensa perder la
robustez de TF-IDF cuando los datos son limitados.

In [ ]:
# Dataset y DataLoaders de PyTorch.

# TODO (Fase 4)

In [ ]:
# Definición del modelo LSTM.

# TODO (Fase 4)

In [ ]:
# Bucle de entrenamiento explícito con macro-F1 por época.

# TODO (Fase 4)

In [ ]:
# Curvas de pérdida y macro-F1 (entrenamiento vs. validación).

# TODO (Fase 4)

In [ ]:
# Evaluación en test.

# TODO (Fase 4)

<!-- LEER: ¿supera a TF-IDF? ¿hay sobreajuste? ¿qué clases abandona primero?
     Las curvas deberían mostrar la pérdida de validación bajando mientras el macro-F1
     se estanca: la firma característica de un modelo que se refugia en la clase mayoritaria. -->

---

## 7. Modelo 3 — BiLSTM con embeddings preentrenados en español

Tres cambios sobre el modelo anterior, cada uno con su razón:

1. **Embeddings preentrenados** (`es_core_news_lg`, 300d): el modelo ya no tiene que aprender
   qué significa «excelente» a partir de 40,000 ejemplos; lo trae de un corpus mucho mayor.
2. **Bidireccionalidad**: en «no estuvo mal» la negación precede al término polar, y una LSTM
   unidireccional llega tarde a esa información.
3. **`pack_padded_sequence`**: los notebooks guía leen `hidden[-1]` sobre secuencias rellenas
   con `[PAD]`, de modo que el estado final corresponde al relleno y no al final del texto.
   Empaquetar las secuencias corrige esto (`docs/DECISIONS.md` §D-002).

Comparamos además dos variantes: embeddings **congelados** frente a **afinados**.

In [ ]:
# Matriz de embeddings desde spaCy + reporte de cobertura.

# TODO (Fase 4)

In [ ]:
# BiLSTM con pack_padded_sequence.

# TODO (Fase 4)

In [ ]:
# Variante A: embeddings congelados.

# TODO (Fase 4)

In [ ]:
# Variante B: embeddings afinados.

# TODO (Fase 4)

In [ ]:
# Comparación de ambas variantes.

# TODO (Fase 4)

<!-- LEER: ¿cuánto aporta cada uno de los tres cambios? ¿congelar o afinar, y por qué
     en este corpus concreto (pista: cobertura del EDA §3.7 y tamaño del conjunto)? -->

---

## 8. Modelo 4 — BETO, un transformer en español

El último escalón: `dccuchile/bert-base-spanish-wwm-cased`, BERT preentrenado sobre
un corpus grande de español. A diferencia de todo lo anterior, las representaciones son
**contextuales**: el vector de «bueno» cambia según lo que lo rodea.

Nos interesa tanto el desempeño como el costo. Un transformer tiene ~110M de parámetros
frente a los pocos millones de la BiLSTM, y ese contraste alimenta el análisis de
costo/beneficio de la §14.

In [ ]:
# Tokenizador de BETO + datasets.

# TODO (Fase 4)

In [ ]:
# Fine-tuning (o extracción de features congeladas si no hay GPU).

# TODO (Fase 4)

In [ ]:
# Evaluación en test + tiempo y número de parámetros.

# TODO (Fase 4)

<!-- LEER: ¿cuánto gana sobre la BiLSTM, y a qué precio en cómputo? ¿mejora sobre todo
     en las clases minoritarias o de forma uniforme? -->

---

# Extensiones

Las cuatro secciones siguientes van más allá de lo que muestran los notebooks guía. Cada una
plantea una pregunta antes de ejecutarse y reporta el resultado aunque contradiga la
hipótesis de partida.

## 9. Extensión A — La polaridad es ordinal, no categórica

**Pregunta.** Todos los modelos anteriores tratan las cinco estrellas como cinco categorías
sin relación entre sí: para la cross-entropy, confundir 4★ con 5★ es tan grave como confundir
1★ con 5★. Pero no lo es. ¿Cambia algo si le decimos al modelo que la escala está ordenada?

Comparamos tres formulaciones sobre la **misma arquitectura**, para que la única variable sea
la salida:

- **(a)** cross-entropy categórica — lo que veníamos haciendo;
- **(b)** regresión a un valor continuo + umbrales optimizados sobre validación;
- **(c)** codificación ordinal acumulativa: cuatro clasificadores binarios «¿es > k?».

In [ ]:
# Formulación (a): cross-entropy (referencia).

# TODO (Fase 5)

In [ ]:
# Formulación (b): regresión + búsqueda de umbrales en validación.

# TODO (Fase 5)

In [ ]:
# Formulación (c): codificación ordinal acumulativa.

# TODO (Fase 5)

In [ ]:
# Comparación: macro-F1, MAE, QWK y % de errores a distancia >= 2.

# TODO (Fase 5)

In [ ]:
# Matrices de confusión de las tres formulaciones, lado a lado.

# TODO (Fase 5)

<!-- LEER: la pregunta clave no es cuál gana en macro-F1, sino si los errores se acercan a
     la diagonal. Un modelo que se equivoca por una estrella es útil; uno que se equivoca
     por cuatro, no. Si el MAE mejora a costa del macro-F1, discutir ese compromiso en lugar
     de esconderlo. -->

## 10. Extensión B — ¿Generaliza el modelo a destinos que nunca vio?

**Pregunta.** El 41% del corpus es Quintana Roo, y solo Tulum e Isla Mujeres son el 36%. Con
una partición aleatoria, el modelo ve en entrenamiento reseñas de los mismos hoteles que
luego evalúa. ¿Está aprendiendo a reconocer sentimiento, o a reconocer destinos?

Entrenamos el mejor modelo dos veces —con el Split A aleatorio y con el Split B geográfico,
donde regiones enteras quedan fuera del entrenamiento— y medimos la diferencia.

**Hipótesis.** El macro-F1 caerá bajo el Split B. Si no cae, también es un resultado, y
significaría que la señal de sentimiento es robusta al destino.

In [ ]:
# Entrenamiento con Split B (regiones held-out).

# TODO (Fase 5)

In [ ]:
# Comparación Split A vs. Split B.

# TODO (Fase 5)

In [ ]:
# Desglose del rendimiento por región held-out.

# TODO (Fase 5)

In [ ]:
# Inspección de errores: ¿se apoyaba el modelo en topónimos y nombres propios?

# TODO (Fase 5)

<!-- LEER: magnitud de la caída, qué regiones sufren más, y qué implica para un\n     despliegue real donde aparecen destinos nuevos constantemente. -->

## 11. Extensión C — ¿En qué se fija el modelo? Atención e interpretabilidad

**Pregunta.** Hasta aquí la BiLSTM comprime toda la reseña en un único vector y no sabemos
qué pesó en la decisión. Añadimos una capa de **atención aditiva** sobre las salidas de la
BiLSTM, que además de (posiblemente) mejorar el desempeño produce un peso por token.

Esto conecta con el EDA: en §3.5 identificamos, mediante log-odds, los términos más
distintivos de cada polaridad. ¿Llega la atención por su cuenta a los mismos términos?

Es también la puerta conceptual a los transformers, que es justo la transición que anuncia
el notebook 3 del curso.

In [ ]:
# BiLSTM con atención aditiva.

# TODO (Fase 5)

In [ ]:
# Evaluación y comparación con la BiLSTM sin atención.

# TODO (Fase 5)

In [ ]:
# Visualización: texto coloreado por peso de atención, una reseña por polaridad.

# TODO (Fase 5)

In [ ]:
# Contraste con el léxico distintivo del EDA §3.5.

# TODO (Fase 5)

<!-- LEER: coincidencias y divergencias con el EDA. Buscar también casos donde la\n     atención se fija en algo inesperado: suele delatar atajos aprendidos. -->

## 12. Extensión D — Anatomía del espacio de embeddings

**Pregunta.** El notebook 3 del curso explora similitud semántica con `lion / cat / pet` y la
aritmética `king - man + woman ≈ queen`. Aquí hacemos lo mismo pero con **vocabulario de
nuestro dominio**, y añadimos una comparación que el guía no hace: enfrentamos los embeddings
**preentrenados** a los que la red **aprendió resolviendo nuestra tarea**.

**Hipótesis.** Los preentrenados organizan el espacio por *tema* (playa cerca de mar, hotel
cerca de habitación); los aprendidos lo organizan por *sentimiento* (excelente cerca de
maravilloso, y ambos lejos de pésimo aunque los tres sean adjetivos del mismo campo).

In [ ]:
# Vecinos más cercanos de términos de dominio: sargazo, alberca, mesero, amabilidad.

# TODO (Fase 5)

In [ ]:
# Los mismos términos en los embeddings aprendidos por la red.

# TODO (Fase 5)

In [ ]:
# Proyección 2D (t-SNE/UMAP) coloreada por polaridad media del término.

# TODO (Fase 5)

In [ ]:
# Aritmética vectorial propia del dominio, análoga al ejemplo del notebook guía.

# TODO (Fase 5)

<!-- LEER: ¿se confirma la hipótesis? Esto explica de forma concreta y visual por qué\n     afinar los embeddings ayudó (o no) en la §7. -->

---

## 13. Tarea de control — predecir el tipo de establecimiento

**Pregunta.** Si el macro-F1 en polaridad resulta modesto, ¿es culpa del modelo, del
preprocesamiento, o de la tarea?

Para responderlo tomamos la mejor arquitectura **sin cambiarle absolutamente nada** y la
ponemos a predecir `Type` (Hotel / Restaurant / Attractive): tres clases razonablemente
balanceadas, no ordinales, sobre exactamente el mismo texto de entrada.

Si el mismo pipeline alcanza un macro-F1 alto aquí, queda demostrado que la dificultad de la
polaridad está en su naturaleza —ordinal, desbalanceada, con ruido de etiqueta— y no en
nuestra implementación.

In [ ]:
# Entrenamiento de la mejor arquitectura sobre Type.

# TODO (Fase 6)

In [ ]:
# Comparación directa: mismo modelo, misma entrada, polaridad vs. tipo.

# TODO (Fase 6)

<!-- LEER: la diferencia de macro-F1 entre ambas tareas es el argumento central de\n     esta sección. Cuantificarla y explicarla. -->

---

## 14. Comparación global y análisis de errores

In [ ]:
# Tabla comparativa: modelo x (macro-F1, accuracy, MAE, QWK, params, tiempo).

# TODO (Fase 6)

In [ ]:
# Gráfica costo (tiempo/parámetros) vs. beneficio (macro-F1).

# TODO (Fase 6)

In [ ]:
# Matrices de confusión de los cuatro modelos, lado a lado.

# TODO (Fase 6)

In [ ]:
# Análisis cualitativo: ejemplos mal clasificados agrupados por causa.

# TODO (Fase 6)

In [ ]:
# Los casos donde TODOS los modelos fallan.

# TODO (Fase 6)

<!-- REDACTAR:
     - Qué aportó realmente cada salto de la escalera, en puntos de macro-F1 y a qué costo.
     - Dónde está el punto de rendimientos decrecientes.
     - Las causas de error: ironía, reseñas mixtas, textos muy cortos, y etiquetas
       incoherentes con el texto (que ponen un techo al desempeño alcanzable y no son
       responsabilidad del modelo).
     - Si todos los modelos fallan en los mismos casos, el problema está en los datos. -->

---

## 15. Conclusiones

<!-- REDACTAR — hallazgos numerados, no un resumen de lo hecho. Cada uno con su evidencia:

     1. Respuesta a la pregunta de la §0.
     2. Qué aporta cada nivel de representación sobre ESTA tarea concreta.
     3. Qué reveló el tratamiento ordinal.
     4. Qué reveló la evaluación geográfica.
     5. Qué reveló el contraste con la tarea de control.

     ### Limitaciones (honestas, no decorativas)
     - Submuestra de 40k, no el corpus completo.
     - Un solo corpus, un solo dominio, una sola variedad del español.
     - Ruido de etiqueta inherente a las reseñas autoasignadas.
     - Exploración de hiperparámetros acotada por el presupuesto de cómputo.

     ### Trabajo futuro
-->